# Chapter 02-06 · Looking at two things at once

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - the arithmetic is trivial, the conclusion is not

**Prerequisites:** 02-05. You should be able to read a distribution and say why an outlier is a
question rather than a category.

**Position in the learning path:** module 02, chapter 6 of 8. Before: **02-05**. After: **02-07**,
on showing what you found without misleading anyone.

---

## Why this matters

Every chapter so far has looked at one column at a time. The moment you look at two, a new class of
mistake opens up - and it is the one that has embarrassed the most analysts.

In this chapter, electric bikes rent out **more often than classic bikes at every single station**,
and **less often overall**. Both statements are arithmetically correct, computed from the same four
numbers. There is no error anywhere.

Then a second demonstration: four datasets with **identical means, identical variances, identical
correlation and the identical fitted line** - which look nothing like each other. A summary
statistic cannot see shape.

Together they make one point: *a relationship is not a property of two columns. It is a property of
two columns and everything you did or did not hold constant.*

## What you will be able to do

By the end of this chapter you can:

1. **Construct and detect** Simpson's paradox, and explain why it happens in terms of group sizes.
2. **Decide** which level of aggregation answers the question - which is a causal judgement, not a
   statistical one.
3. **Demonstrate** that correlation, means and a fitted line can be identical across completely
   different data.
4. **Read** a scatter matrix and a colour-coded scatter to find a lurking third variable.
5. **State** what a correlation coefficient cannot see.

## Warm-up: retrieve, do not reread

From memory:

1. What are the three kinds of outlier?
2. Why did the three-sigma rule delete every festival day?
3. What does the mean answer that the median does not?
4. Name three costs of a log transform.

<br>

*Answers: (1) an error, rare but real, a different population. (2) it assumes normality, and the
standard deviation was inflated by the very points being tested. (3) total divided by count - a
statement about the total rather than about a typical case. (4) zeros and negatives undefined;
coefficients become percentages; back-transforming gives a median rather than a mean.*

## The situation

Maria trialled electric bikes for a month at two stations, alongside her classic ones. She wants to
know which type rents out more reliably, so she can decide what to buy.

Four numbers per row: how many were put out, and how many were rented.

**The question this chapter answers:** when the same data gives two opposite answers, which one is
the answer?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

trial = pd.DataFrame({
    "station": ["north", "north", "south", "south"],
    "bike":    ["electric", "classic", "electric", "classic"],
    "rented":  [81, 234, 192, 55],
    "offered": [87, 270, 263, 80],
})
trial["rate"] = (trial["rented"] / trial["offered"]).round(3)
trial

### Predict before running

Work these out with a pen before running the next cell. It takes a minute and it is the whole point
of the chapter.

- **North:** electric `81 / 87`, classic `234 / 270`.
- **South:** electric `192 / 263`, classic `55 / 80`.
- **Overall:** electric `(81 + 192) / (87 + 263)`, classic `(234 + 55) / (270 + 80)`.

1. Which bike type wins at north?
2. Which wins at south?
3. Which wins overall?

In [ ]:
by_station = trial.pivot_table(index="station", columns="bike", values="rate")
overall = trial.groupby("bike")[["rented", "offered"]].sum()
overall["rate"] = (overall["rented"] / overall["offered"]).round(3)

print("RATE BY STATION")
print(by_station.to_string())
print("\nOVERALL")
print(overall.to_string())

## Failure lab: the answer that reverses

| | classic | electric | winner |
|---|---|---|---|
| **north** | 0.867 | **0.931** | electric |
| **south** | 0.688 | **0.730** | electric |
| **overall** | **0.826** | 0.780 | **classic** |

**Electric bikes rent out more often at north. They rent out more often at south. And they rent out
less often overall.**

Both bike types were offered exactly 350 times, so this is not a sample-size artefact. Every number
is a correct division. Nothing is missing.

This is **Simpson's paradox**, and once you see the mechanism it stops being paradoxical.

### Why it happens

Look at *where* each type was placed.

In [ ]:
placement = trial.pivot_table(index="bike", columns="station", values="offered")
placement["% at south"] = (placement["south"] / placement.sum(axis=1) * 100).round(0)
print(placement.to_string())
print(f"\nnorth rents {by_station.mean(axis=1)['north']:.1%} of what it offers; "
      f"south rents {by_station.mean(axis=1)['south']:.1%}")

There it is. **South is the quieter station** - both bike types do worse there - and **75% of the
electric bikes were sent to south**, while 77% of the classic bikes stayed at north.

So the overall electric rate is mostly a measurement of *south*, and the overall classic rate is
mostly a measurement of *north*. Comparing them compares the stations, not the bikes.

The overall number is a **weighted average**, and the two types have different weights:

- electric: `(0.931 x 87 + 0.730 x 263) / 350` - three quarters weighted to the low-performing station
- classic: `(0.867 x 270 + 0.688 x 80) / 350` - three quarters weighted to the high-performing one

**Station is a lurking variable**: it affects the outcome *and* it is unevenly distributed across
the groups being compared. That is exactly the definition of a confounder from 00-04, arriving here
through where somebody parked the bikes.

### Which answer is right?

This is the part that matters, and it is not a statistical question.

**If station affected which bike went where** - the trial team put electric bikes at south because
there was space, or because south's racks fit them - then station is a **confounder**, the groups are
not comparable, and the **within-station comparison is the honest one. Electric wins.**

**If the bike type caused the placement in a way that is part of its effect** - suppose electric
bikes must go where there is power, and being near power is genuinely part of what an electric bike
is - then station is on the causal path, it is a **mediator**, and controlling for it removes part of
the effect you care about. Then the **overall number is the right one.**

The data is identical in both cases. **Only knowing how the bikes were assigned tells you which
story you are in**, and that is a provenance question (02-02), not a calculation.

You met this exact fork in 00-04's E14, where adjusting for a mediator turned a correct total effect
of 6.97 into 0.97. Simpson's paradox is the same fork, seen from the aggregation side.

### What to do about it

| | |
|---|---|
| **Always look at the breakdown before quoting an aggregate** | One `pivot_table`. It is how you find out a paradox exists at all |
| **Ask how the groups came to be unequal** | If assignment was not random, expect a confounder |
| **Report both levels when they disagree** | "Electric wins at each station and loses overall, because three quarters of them were at the quieter station" is one true sentence |
| **Prefer a designed comparison** | Equal numbers of each type at each station makes the paradox arithmetically impossible |
| **Never let a dashboard aggregate silently** | Most Simpson's paradoxes reach a decision through an average someone did not know was weighted |

**The reassuring part:** the paradox requires unequal group sizes. Had 175 of each type been placed
at each station, the overall comparison and the per-station comparisons would agree by construction.
Balance removes it entirely, which is why randomised assignment (00-04) is worth so much.

---

## Failure lab 2: four datasets, one set of statistics

A second way two columns can deceive you, and this one is a century-old classic.

Below are four small datasets. Before plotting them, we compute for each: the mean of x, the mean of
y, the variance of each, the correlation, and the fitted regression line.

**Predict before running:** how similar will those five numbers be across the four datasets?

In [ ]:
# Anscombe's quartet (F. J. Anscombe, 1973). Four datasets, chosen to make exactly this point.
x_common = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = {
    "I":   (x_common, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x_common, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x_common, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
            [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

summary = []
for name, (x, y) in quartet.items():
    x, y = np.array(x, float), np.array(y, float)
    slope, intercept = np.polyfit(x, y, 1)
    summary.append({"set": name, "mean x": x.mean(), "mean y": y.mean(),
                    "var x": x.var(ddof=1), "var y": y.var(ddof=1),
                    "correlation": np.corrcoef(x, y)[0, 1],
                    "fitted line": f"y = {intercept:.2f} + {slope:.2f}x"})
pd.DataFrame(summary).set_index("set").round(2)

**Every number is the same.** Same means, same variances, correlation 0.82 in all four, and the
identical fitted line `y = 3.00 + 0.50x`.

Any report built on those statistics would describe the four datasets as interchangeable. Now look
at them.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharex=True, sharey=True)
line_x = np.array([3, 20])

for ax, (name, (x, y)) in zip(axes, quartet.items()):
    ax.scatter(x, y, s=45, color="#0072B2", zorder=3)
    ax.plot(line_x, 3.0 + 0.5 * line_x, color="#D55E00", linewidth=1.6)
    ax.set_title(f"set {name}   r = 0.82")
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("Identical means, variances, correlation and fitted line", y=1.04)
fig.tight_layout()
plt.show()

### Diagnosis

- **Set I** is what everybody pictures when they hear "correlation 0.82": a linear relationship with
  scatter.
- **Set II** is a clean **curve**. The relationship is perfect and not linear at all, so a straight
  line is exactly the wrong model - and the correlation cannot tell you.
- **Set III** is a *perfect* straight line plus **one outlier**, which drags the fitted line off the
  true relationship. Remove that point and the line changes completely.
- **Set IV** is a **vertical stack at x = 8 plus one point at x = 19**. There is no relationship
  being measured at all: a single observation determines the entire slope. Delete it and the slope
  is undefined.

**What a correlation coefficient cannot see:**

| It cannot see | Set |
|---|---|
| Curvature - it measures *linear* association only | II |
| The influence of a single point | III, IV |
| Whether the relationship exists across the range, or rests on one leverage point | IV |
| Clusters, gaps, ceilings, floors, or two populations | any |

**The rule, and it is the oldest rule in the subject:**

> **Plot it.** A correlation, a mean and a fitted line are summaries, and a summary is a thing you
> compute *after* you have seen the shape - never instead.

This is also why 05-05 spends a chapter on residual plots. A model's fitted line looks the same in
all four cases; only the pattern of what it got wrong distinguishes them.

## Looking at more than two

With three or more columns, two tools do most of the work. Neither is sophisticated, and both are
worth reaching for before any model.

In [ ]:
rng = np.random.default_rng(4)
n = 400
temp = rng.normal(18, 6, n)
weekend = rng.random(n) < 2 / 7
# SYNTHETIC: rentals rise with temperature, and weekends add a large flat boost.
rentals = 40 + 3.0 * temp + 45 * weekend + rng.normal(0, 8, n)
staff = 6 + 0.05 * rentals + rng.normal(0, 1.2, n)          # staffing follows demand

park = pd.DataFrame({"temp_c": temp.round(1), "weekend": weekend.astype(int),
                     "rentals": rentals.round(), "staff_on_duty": staff.round(1)})
print(park.corr(numeric_only=True).round(2).to_string())

In [ ]:
columns = ["temp_c", "rentals", "staff_on_duty"]
fig, axes = plt.subplots(len(columns), len(columns), figsize=(8, 8))

for i, row in enumerate(columns):
    for j, col in enumerate(columns):
        ax = axes[i, j]
        if i == j:
            ax.hist(park[col], bins=25, color="#0072B2")
        else:
            for value, colour in [(0, "#0072B2"), (1, "#D55E00")]:
                m = park["weekend"] == value
                ax.scatter(park.loc[m, col], park.loc[m, row], s=6, alpha=0.5, color=colour)
        if i == len(columns) - 1:
            ax.set_xlabel(col, fontsize=8)
        if j == 0:
            ax.set_ylabel(row, fontsize=8)
        ax.tick_params(labelsize=6)

fig.suptitle("Scatter matrix, coloured by weekend (orange) vs weekday (blue)", y=0.995)
fig.tight_layout()
plt.show()

Two things are visible in that picture that the correlation table cannot show.

**The rentals histogram has two humps** - the weekday cluster and the weekend cluster - exactly the
"two populations in one column" signature from 02-05. The correlation between temperature and
rentals is a single number averaged across both.

**Colour separates the clouds.** In the temperature-versus-rentals panel, the orange points sit as a
parallel band above the blue ones. The *slope* is the same in both - temperature does the same thing
on both kinds of day - but the *level* differs. That is an additive effect of a third variable, and
seeing it in a picture tells you immediately that `weekend` belongs in the model as its own feature
rather than as an interaction.

**And `staff_on_duty` correlates strongly with rentals for a reason that should stop you.** Staff
are rostered *because* demand is expected. Used as a feature to predict rentals it would work
beautifully and be useless - you do not know tomorrow's roster before you decide tomorrow's roster,
and increasing staff does not increase demand. It is 00-04's "great predictor, useless lever", and a
scatter matrix is where you notice it.

**The two habits from this section:**

1. **Colour by a candidate third variable.** It costs one line and it is how lurking variables are
   found.
2. **Look at every strong correlation and ask which direction the causation runs** - and whether the
   variable will exist at prediction time.

## Common misconceptions

**"Correlation measures how related two things are."**
It measures how close they are to a *straight line*. Anscombe's set II has a perfect relationship
and a correlation of 0.82; a perfect U-shape can have a correlation of exactly zero.

**"A high correlation means a strong relationship."**
Set IV has correlation 0.82 and rests entirely on one point. Correlation says nothing about how many
observations support it.

**"Simpson's paradox is a curiosity."**
It has decided court cases, medical guidance and university admissions policy. Any time a rate is
compared between groups that were not equally distributed across a third variable, it is available.

**"If the subgroups disagree with the total, the subgroups are right."**
Usually, but not always - it depends on whether the third variable is a confounder or a mediator, and
that is a question about how the world works. 00-04, 12-07.

**"More variables always give a clearer picture."**
A scatter matrix of forty columns is 1,600 panels and shows nothing. Look at the handful you have a
reason to look at, and use the rest for modelling.

**"The correlation matrix is a good first look."**
It is a good *second* look. It compresses each relationship to one number that cannot see curvature,
outliers, clusters or leverage - which is precisely what a first look is for.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-06_multivariate_solutions.ipynb`.

### Quick understanding

**E1 (define).** State Simpson's paradox in one sentence, and name the condition without which it
cannot occur.

**E2 (explain).** Anscombe's set II has a perfect relationship and a correlation of 0.82. Explain
why those two facts are compatible.

**E3 (explain).** Why is `staff_on_duty` a bad feature for predicting rentals, even though it
correlates strongly?

### Hand calculation

**E4 (calculate).** With a pen, from the trial table: compute both bike types' rates at each station
and overall. Then recompute the overall rates *as if* each type had been offered 175 times at each
station, keeping the per-station rates unchanged. Which type wins now, and what does that tell you?

**E5 (calculate).** A hospital reports that surgery A has a 78% success rate and surgery B has 83%.
Surgery A was performed on 200 severe cases (70% success) and 100 mild ones (94%). Surgery B was
performed on 50 severe (60%) and 250 mild (88%). Verify both overall rates, then say which surgery
you would want and why.

### Coding

**E6 (code).** Write `simpsons_check(df, group, category, numerator, denominator)` that reports the
rate per category within each group and overall, and prints a warning when the overall winner
differs from the within-group winner. Test it on the trial data.

**E7 (code).** Compute the correlation between `temp_c` and `rentals` overall, then separately for
weekdays and weekends. Explain why the three numbers differ and which one you would report.

### Interpretation

**E8 (interpret).** A colleague shows a correlation matrix and says the two strongest pairs are the
important relationships. Give three reasons to be careful, referring to specific Anscombe sets.

### Debugging

**E9 (diagnose).** A department reports that its new onboarding process improved 30-day retention
from 61% to 68%. A sceptic points out that the new process was rolled out first to enterprise
customers. Describe the analysis you would run, what you expect to find, and what would convince you
the improvement is real.

### Exam and interview reasoning

**E10 (defend).** *"Our A/B test shows variant B is better overall, but variant A is better in every
country. Which do we ship?"* Answer in about 130 words.

**E11 (design).** You are given a dataset with 60 columns and asked to "explore it". Describe your
first thirty minutes, in order, and say what you would deliberately not do.

### Transfer to a different situation

**E12 (design).** For each, name the lurking variable you would look for first:
(a) a university department appears to admit men at a higher rate than women overall;
(b) a hospital with a higher death rate than its neighbour;
(c) a marketing channel with a lower cost per acquisition than every other channel;
(d) a school whose exam results fell after a new curriculum;
(e) a delivery route with an unusually high damage rate.

### Explain it to someone non-technical

**E13 (explain).** In under 80 words, explain to Maria how electric bikes can rent out better at
both stations and worse overall, without using the words "average", "weighted" or "paradox".

### Optional challenge

**E14 (code + diagnose).** Build a dataset where a relationship **reverses sign** rather than merely
weakening: overall, more staff on duty predicts *fewer* rentals per bike, while within every station
more staff predicts *more*. Show both fits, plot the reversal, and say which fit answers "should we
roster more staff?".

In [ ]:
# Your workspace. Still in memory: trial, by_station, overall, quartet, park.

## Mastery check

Without scrolling up, can you:

- [ ] State Simpson's paradox and the condition it requires? *(If not: "Why it happens".)*
- [ ] Say what decides which level of aggregation is correct? *(If not: "Which answer is right?".)*
- [ ] Name four things a correlation coefficient cannot see? *(If not: "Failure lab 2".)*
- [ ] Say what colouring a scatter by a third variable is for? *(If not: "Looking at more than two".)*
- [ ] Explain why a strongly correlated feature can still be useless? *(If not: the `staff_on_duty`
      paragraph.)*

## What should now feel instinctive

1. **Break the aggregate down before quoting it.** One `pivot_table`, and it is how you learn a
   paradox is present at all.
2. **Ask how the groups came to be unequal.** Unequal allocation plus an effect on the outcome is
   the whole recipe.
3. **Plot it before you summarise it.** Four datasets, one set of statistics.
4. **Colour by a candidate third variable** whenever a relationship looks too clean or too messy.
5. **For every strong correlation, ask which way the causation runs and whether the feature exists
   at prediction time.**

## Flashcards

| Question | Answer |
|---|---|
| Simpson's paradox | A comparison that holds in every subgroup can reverse when the subgroups are pooled |
| What does it require? | Unequal group sizes, and a third variable that affects the outcome |
| Which level is correct? | Depends on whether the third variable is a confounder (use subgroups) or a mediator (use the total) - a causal question |
| How do you make it impossible? | Balance the groups - equal allocation, ideally randomised |
| What does correlation measure? | Closeness to a straight line, nothing else |
| Four things correlation cannot see | Curvature, outliers, leverage points, clusters |
| Anscombe's quartet | Four datasets with identical means, variances, correlation and fitted line, and completely different shapes |
| The oldest rule in the subject | Plot it before you summarise it |
| What is colouring a scatter by a third variable for? | Finding lurking variables and separate populations |
| Why can a strong correlation be a useless feature? | It may be a consequence rather than a cause, or unavailable at prediction time |

## Next

**02-07 · Honest visualisation, correlation versus causation, and the limits of EDA.**

You now have the tools to find structure in data and a healthy suspicion of what they show. The next
chapter is about the last step: presenting what you found without persuading anyone of something you
did not establish - and being clear about what exploratory analysis can and cannot settle. It closes
the theory half of module 02, and then 02-08 puts all six chapters on one real dataset.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).